# Lecture 5: Arrays & Scientific Computing
**BME Computation Course · Interactive lecture · 75 minutes**

Big question: **How do we represent vectors and matrices and calculate with their elements?**

Prerequisites: variables, arithmetic, comparisons, loops, and lists. All data below are synthetic classroom examples. Thresholds are exercise rules, not clinical criteria.

Open in Jupyter or Google Colab. Only NumPy is required; no downloads or data files are needed. If the import fails, install NumPy in your notebook environment (for example, run `%pip install numpy` in a separate cell). Run cells from top to bottom. Pause to predict before running each demonstration.

By the end, you should be able to:
- Create a numerical array and explain its shape, dimensions, and data type.
- Select elements, subvectors, rows, columns, and submatrices.
- Replace a loop over independent measurements with an array expression.
- Filter measurements using a Boolean mask.
- Express row and column reductions using axis indices and summation notation.
- Recognize common mistakes involving units, shapes, and shared data.

Lecture route: motivation (8 min), array basics (10), indexing (12), vectorization (12), filtering (10), two-dimensional data (13), practice and exit check (10). Optional sections follow the exit check.


In [1]:
import numpy as np
np.set_printoptions(precision=2, suppress=True)
print("NumPy version:", np.__version__)


NumPy version: 2.4.6




## 1. Why arrays? Vectors and matrices in computation (8 min)
A vector stores an ordered sequence of elements; a matrix stores elements indexed by row and column. NumPy arrays let us represent these objects and compute with them directly. For example, a sampled signal can be represented by a vector and a grayscale image by a matrix.

For a vector $x$, scalar multiplication by $a$ gives $y_i = a x_i$. NumPy writes this as `y = a * x`. A Python list uses different multiplication rules.

**Predict:** How do the results of multiplying a list and a NumPy array by 2 differ?


In [2]:
temperature_list_c = [36.5, 37.0, 38.0, 36.8]
print("List * 2:", temperature_list_c * 2)

temperature_c = np.array(temperature_list_c)
print("Array * 2:", temperature_c * 2)


List * 2: [36.5, 37.0, 38.0, 36.8, 36.5, 37.0, 38.0, 36.8]
Array * 2: [73.  74.  76.  73.6]


The list repeats its contents. The array performs **scalar multiplication**: each element is multiplied by 2.

We use one-dimensional arrays to represent vectors and two-dimensional arrays to represent matrices. Arrays store numerical values; units such as °C or mV must be recorded separately.

**Ask:** If $x$ has $n$ elements, how many elements does $2x$ have?


## 2. Array shape and dimension (10 min)
`np` is the conventional alias for NumPy. An array's **shape** specifies its length along each axis.

| Object | NumPy shape | `ndim` | `size` |
|---|---|---:|---:|
| Scalar | `()` | 0 | 1 |
| Vector with $n$ elements | `(n,)` | 1 | $n$ |
| Matrix with $m$ rows and $n$ columns | `(m, n)` | 2 | $mn$ |
| Explicit row vector | `(1, n)` | 2 | $n$ |
| Explicit column vector | `(n, 1)` | 2 | $n$ |

**Terminology:** A vector in $\mathbb{R}^n$ has $n$ components, but its one-dimensional NumPy representation has `ndim == 1`. NumPy's `ndim` counts axes, not vector components. Shape `(n,)` has no explicit row or column orientation.

`dtype` specifies the data type shared by the elements. Use `dtype=float` when elements may contain fractional values.


In [3]:
print("Values:", temperature_c)
print("Dimensions:", temperature_c.ndim)
print("Shape:", temperature_c.shape)
print("Number of values:", temperature_c.size)
print("Data type:", temperature_c.dtype)

sample_numbers = np.arange(6)  # 0 through 5; stop is excluded
sample_times_s = np.linspace(0, 2.5, 6)  # 6 evenly spaced times; includes both ends
empty_signal_mv = np.zeros(6)
print("Sample numbers:", sample_numbers)
print("Sample times (s):", sample_times_s)
print("Initialized signal (mV):", empty_signal_mv)


Values: [36.5 37.  38.  36.8]
Dimensions: 1
Shape: (4,)
Number of values: 4
Data type: float64
Sample numbers: [0 1 2 3 4 5]
Sample times (s): [0.  0.5 1.  1.5 2.  2.5]
Initialized signal (mV): [0. 0. 0. 0. 0. 0.]


**Predict and discuss:** How many entries will `np.arange(1, 5)` have? How is that different from `np.linspace(1, 5, 5)`? Prefer integer sample numbers or `linspace` when you need a specified number of time points; floating-point steps in `arange` can make endpoints surprising.


## 3. Indexing and slicing (12 min)
Let $t$ be a vector of sample times and $x$ a vector of voltages, both with length 8. The element $x_i$ is the voltage at time $t_i$.

We use **zero-based indices** throughout: $x_0, x_1, \ldots, x_7$.

- `x[i]` selects the scalar element $x_i$; `x[-1]` selects the last element.
- `x[a:b]` selects elements with indices $a \leq i < b$.
- `x[a:b:s]` selects indices $a, a+s, a+2s, \ldots$ less than $b$, for positive step $s$.
- With a positive step, omitted bounds default to the beginning and end; the default step is 1.

**Predict:** Which elements and sample times are selected by `signal_mv[2:5]`?


In [4]:
time_s = np.arange(8) * 0.5
signal_mv = np.array([0.1, 0.2, 0.8, 1.2, 0.7, 0.3, 0.2, 0.1])
print("Times (s):", time_s)
print("Voltages (mV):", signal_mv)
print("First / last voltage (mV):", signal_mv[0], signal_mv[-1])
print("Window times (s):", time_s[2:5])
print("Window voltages (mV):", signal_mv[2:5])
print("Every second voltage (mV):", signal_mv[::2])


Times (s): [0.  0.5 1.  1.5 2.  2.5 3.  3.5]
Voltages (mV): [0.1 0.2 0.8 1.2 0.7 0.3 0.2 0.1]
First / last voltage (mV): 0.1 0.1
Window times (s): [1.  1.5 2. ]
Window voltages (mV): [0.8 1.2 0.7]
Every second voltage (mV): [0.1 0.8 0.7 0.2]


### Pause and try A
1. Select the last three voltages and their corresponding times.
2. Select readings at times 0.5, 1.0, and 1.5 seconds using a slice.
3. Explain why `signal_mv[1:3]` returns two readings.

Selecting every second reading demonstrates indexing; it is not a complete signal downsampling method, which can require filtering.


In [5]:
# Write your selections here.


## 4. Elementwise operations and vectorization (12 min)
For vectors $x,y \in \mathbb{R}^n$ and scalar $a$, NumPy applies these operations to corresponding elements:

| NumPy expression | Definition of output element $z_i$ |
|---|---|
| `x + y` | $z_i = x_i + y_i$ |
| `x - y` | $z_i = x_i - y_i$ |
| `a * x` | $z_i = a x_i$ |
| `x * y` | $z_i = x_i y_i$ (elementwise product) |
| `x / y` | $z_i = x_i/y_i$, assuming $y_i \ne 0$ |
| `x ** 2` | $z_i = x_i^2$ |
| `x + a` | $z_i = x_i+a$ (scalar addition to every element) |

**Vectorization** expresses these component formulas as array operations without an explicit Python loop.

For temperature conversion, $f_i = \frac{9}{5}c_i + 32$. Compare its loop and vectorized implementations. Predict the result for $c_i=37$ °C.


In [6]:
temperature_f_loop = []
for value_c in temperature_c:
    temperature_f_loop.append(value_c * 9 / 5 + 32)

temperature_f = temperature_c * 9 / 5 + 32
print("Loop result (°F):", temperature_f_loop)
print("Array result (°F):", temperature_f)
print("Results agree:", np.allclose(temperature_f_loop, temperature_f))

# Corresponding elements of the two vectors form a pair.
before_bpm = np.array([70, 75, 80, 65], dtype=float)
after_bpm = np.array([82, 81, 96, 78], dtype=float)
change_bpm = after_bpm - before_bpm
percent_change = 100 * change_bpm / before_bpm
print("Changes (bpm):", change_bpm)
print("Percent changes (%):", percent_change)


Loop result (°F): [np.float64(97.7), np.float64(98.6), np.float64(100.4), np.float64(98.24)]
Array result (°F): [ 97.7   98.6  100.4   98.24]
Results agree: True
Changes (bpm): [12.  6. 16. 13.]
Percent changes (%): [17.14  8.   20.   20.  ]


### Shape requirements
For equal-shaped arrays, arithmetic pairs elements with the same indices. If $A,B \in \mathbb{R}^{m\times n}$, then `(A + B)[i, j]` equals $A_{ij}+B_{ij}$ and the result has shape `(m, n)`.

NumPy also permits **broadcasting**. To check whether two shapes are compatible:
1. Align the shapes on the right; treat missing leading axes as length 1.
2. Each pair of axis lengths must be equal, or one must be 1.
3. Equal lengths remain unchanged; an axis of length 1 expands to the other length.

| Input shapes | Result shape | Interpretation |
|---|---|---|
| `(4,)` and `(4,)` | `(4,)` | Corresponding elements |
| `(4,)` and `()` | `(4,)` | Vector and scalar |
| `(3, 4)` and `(4,)` | `(3, 4)` | Same length-4 vector applied to each row |
| `(3, 4)` and `(3, 1)` | `(3, 4)` | One scalar per row applied across its columns |
| `(3, 4)` and `(3,)` | Incompatible | Rightmost lengths 4 and 3 differ; neither is 1 |

Shape compatibility is a computational requirement. In an application, corresponding elements must also represent quantities that can meaningfully be combined; addition and subtraction require compatible units.

**Elementwise versus matrix multiplication:** `A * B` multiplies corresponding elements (with broadcasting if needed). For two matrices, `A @ B` requires shapes `(m, n)` and `(n, p)` and produces shape `(m, p)`, with entries $C_{ij}=\sum_{k=0}^{n-1}A_{ik}B_{kj}$. We focus on elementwise operations today.

**Connection to CA 3:** A recurrence such as $x_{k+1}=0.8x_k+5$ defines each term from the previous term. A loop is a natural implementation of that dependency.

### Pause and try B
Apply the affine transformation $y_i=2(x_i-0.1)$ to the voltage vector using `(signal_mv - 0.1) * 2`. The offset is 0.1 mV and the gain is dimensionless. Store the result, print its units, and verify its first element by hand.


In [7]:
# Write your calibration calculation here.


## 5. Boolean arrays and selection (10 min)
For a vector $x$ and threshold $c$, the comparison `x > c` creates a Boolean vector $b$ of the same shape, where $b_i$ is True exactly when $x_i>c$.

`x[b]` returns the selected elements in index order. For this one-dimensional selection, `b` must have the same length as `x`; the result has length equal to the number of True elements.

Use $c=0.5$ mV. Apply the same Boolean vector to $t$ to obtain the corresponding sample times.


In [8]:
above_threshold = signal_mv > 0.5
print("Mask:", above_threshold)
print("Selected voltages (mV):", signal_mv[above_threshold])
print("Selected times (s):", time_s[above_threshold])
print("Number of selected samples:", np.count_nonzero(above_threshold))

in_window = (time_s >= 1.0) & (time_s < 2.5)
print("Voltages from 1.0 s up to but excluding 2.5 s (mV):", signal_mv[in_window])
print("Any above threshold?", np.any(above_threshold))
print("All above threshold?", np.all(above_threshold))


Mask: [False False  True  True  True False False False]
Selected voltages (mV): [0.8 1.2 0.7]
Selected times (s): [1.  1.5 2. ]
Number of selected samples: 3
Voltages from 1.0 s up to but excluding 2.5 s (mV): [0.8 1.2 0.7]
Any above threshold? True
All above threshold? False


For elementwise conditions use `&` (and), `|` (or), and `~` (not), with each comparison in parentheses. Python's `and` and `or` do not combine arrays element by element. `if signal_mv > 0.5:` is ambiguous for this multi-entry array; first decide whether you mean **any**, **all**, or a selection of entries.

**Ask:** What changes if the threshold uses `>=`? Does the number of selected samples equal the number of separate peaks? Why not?


## 6. Matrices: rows, columns, and submatrices (13 min)
Let $A \in \mathbb{R}^{3\times4}$:

$$
A=\begin{bmatrix}
70 & 80 & 90 & 80\\
60 & 70 & 80 & 70\\
80 & 90 & 100 & 90
\end{bmatrix}.
$$

$A_{ij}$ denotes the element in row $i$ and column $j$, with $i\in\{0,1,2\}$ and $j\in\{0,1,2,3\}$.

We store $A$ as `heart_rate_bpm`; its elements are synthetic measurements in bpm. The indexing rules depend on the matrix shape, not on the application.

- `A[i, j]`: scalar element $A_{ij}$.
- `A[i, :]`: row $i$, returned with shape `(4,)`.
- `A[:, j]`: column $j$, returned with shape `(3,)`.
- `A[:2, :2]`: upper-left $2\times2$ submatrix.

Selecting a single row or column with an integer index removes that axis. Slicing preserves it: `A[:, j:j+1]` has shape `(3, 1)`.


In [9]:
heart_rate_bpm = np.array([
    [70, 80, 90, 80],
    [60, 70, 80, 70],
    [80, 90, 100, 90],
], dtype=float)
print("Matrix shape:", heart_rate_bpm.shape)
print("Element A[1, 2] (bpm):", heart_rate_bpm[1, 2])
print("Row 0 (bpm):", heart_rate_bpm[0, :])
print("Column 1 (bpm):", heart_rate_bpm[:, 1])
print("Upper-left 2 × 2 submatrix (bpm):", heart_rate_bpm[:2, :2], sep="\n")


Matrix shape: (3, 4)
Element A[1, 2] (bpm): 80.0
Row 0 (bpm): [70. 80. 90. 80.]
Column 1 (bpm): [80. 70. 90.]
Upper-left 2 × 2 submatrix (bpm):
[[70. 80.]
 [60. 70.]]


### Reductions: sums and means along an axis
A **reduction** combines elements along a specified axis. For an $m\times n$ matrix $A$:

| Expression | Formula | Output shape |
|---|---|---|
| `np.mean(A, axis=0)` | $c_j=\frac{1}{m}\sum_{i=0}^{m-1} A_{ij}$ | `(n,)`: one mean per column |
| `np.mean(A, axis=1)` | $r_i=\frac{1}{n}\sum_{j=0}^{n-1} A_{ij}$ | `(m,)`: one mean per row |
| `np.mean(A)` | $\mu=\frac{1}{mn}\sum_{i=0}^{m-1}\sum_{j=0}^{n-1} A_{ij}$ | `()`: scalar |

`axis=0` reduces the row index $i$; `axis=1` reduces the column index $j$. By default, the reduced axis is removed. `np.sum` uses the same axis convention without dividing by the number of elements.

**Predict:** For our $3\times4$ matrix, what are the output shapes? Calculate the mean of row 0 by hand.


In [10]:
column_means_bpm = np.mean(heart_rate_bpm, axis=0)
row_means_bpm = np.mean(heart_rate_bpm, axis=1)
print("Column means (bpm):", column_means_bpm, column_means_bpm.shape)
print("Row means (bpm):", row_means_bpm, row_means_bpm.shape)
print("Row maxima (bpm):", np.max(heart_rate_bpm, axis=1))
print("Overall minimum / maximum (bpm):", np.min(heart_rate_bpm), np.max(heart_rate_bpm))


Column means (bpm): [70. 80. 90. 80.] (4,)
Row means (bpm): [80. 70. 90.] (3,)
Row maxima (bpm): [ 90.  80. 100.]
Overall minimum / maximum (bpm): 60.0 100.0


## 7. Integrating the ideas (10 min)
Using $A=$ `heart_rate_bpm`, complete the following without a loop:
1. Extract the first and last columns as one-dimensional arrays $u$ and $v$.
2. Calculate the difference vector $d=v-u$, so $d_i=A_{i,3}-A_{i,0}$.
3. Select the elements satisfying $d_i\geq10$ bpm and count them.
4. Calculate the arithmetic mean $\bar d=\frac{1}{3}\sum_{i=0}^{2}d_i$.
5. Explain how $\bar d$ differs from the mean of all elements of $A$.

State each output's shape before computing. Predict the values from the matrix, then write the NumPy expressions.


In [11]:
# Independent practice: write your code here.


### Exit check
- How many rows, columns, and elements are in an array with shape `(3, 4)`?
- What are the shapes of `A[0, :]`, `A[:, 0]`, and `A[:, 0:1]`?
- Which axis gives one mean per row? Write its summation formula.
- Can arrays of shapes `(3, 4)` and `(4,)` be added? What about `(3, 4)` and `(3,)`? Explain using the broadcasting rule.
- How do `A * B` and `A @ B` differ?

**Core lecture ends here.** The following demonstrations can be used if time permits or in a second session.


## Optional extension 1: A slice can share data (5 min)
Basic slices are **views** of the original array. Changing the slice can change the original. Use `.copy()` when you want independent data. This demonstration creates its own data so the lecture dataset stays intact.


In [12]:
original = np.array([1.0, 2.0, 3.0, 4.0])
window = original[1:3]
window[0] = 99
print("Original after changing view:", original)

independent_window = original[1:3].copy()
independent_window[0] = -5
print("Independent copy:", independent_window)
print("Original after changing copy:", original)


Original after changing view: [ 1. 99.  3.  4.]
Independent copy: [-5.  3.]
Original after changing copy: [ 1. 99.  3.  4.]


## Optional extension 2: Subtracting a column by broadcasting (7 min)
For $A\in\mathbb{R}^{3\times4}$, define $b_i=A_{i,0}$ and $C_{ij}=A_{ij}-b_i$.

Store $b$ with shape `(3, 1)` using `A[:, 0:1]`. In `A - b`, the row lengths agree (3 and 3), and the column of length 1 broadcasts to length 4. The result has shape `(3, 4)`.

In contrast, `A[:, 0]` has shape `(3,)`. When aligned on the right with `(3, 4)`, lengths 3 and 4 are incompatible. A one-dimensional array is not automatically interpreted as a column vector.


In [13]:
first_column_bpm = heart_rate_bpm[:, 0:1]
difference_matrix_bpm = heart_rate_bpm - first_column_bpm
print("Matrix shape:", heart_rate_bpm.shape)
print("Column shape:", first_column_bpm.shape)
print("C[i, j] = A[i, j] - A[i, 0] (bpm):", difference_matrix_bpm, sep="\n")


Matrix shape: (3, 4)
Column shape: (3, 1)
C[i, j] = A[i, j] - A[i, 0] (bpm):
[[ 0. 10. 20. 10.]
 [ 0. 10. 20. 10.]
 [ 0. 10. 20. 10.]]


## Optional extension 3: Missing measurements and data types (5 min)
`np.nan` represents a missing numerical value. Ordinary `mean` propagates it; `nanmean` ignores it. Ignoring missingness is a deliberate analytical choice, not automatic data cleaning. Report the number of observed measurements and consider why values are missing. An all-missing group has no defined observed mean.

Integer arrays cannot retain fractional values assigned into them; choose a floating-point dtype for fractional measurements.


In [14]:
measurements = np.array([70.0, np.nan, 80.0, 75.0])
print("Ordinary mean:", np.mean(measurements))
print("Observed-value mean (bpm):", np.nanmean(measurements))
print("Observed count:", np.count_nonzero(~np.isnan(measurements)))

integer_values = np.array([1, 2, 3])
integer_values[0] = 1.8
float_values = np.array([1, 2, 3], dtype=float)
float_values[0] = 1.8
print("Assigned into integer array:", integer_values)
print("Assigned into floating-point array:", float_values)


Ordinary mean: nan
Observed-value mean (bpm): 75.0
Observed count: 3
Assigned into integer array: [1 2 3]
Assigned into floating-point array: [1.8 2.  3. ]


## Reference and next steps
NumPy documentation: [beginner guide](https://numpy.org/doc/stable/user/absolute_beginners.html), [indexing](https://numpy.org/doc/stable/user/basics.indexing.html), and [broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html).

Next applications: plotting signals, analyzing repeated measurements, processing image regions, and loading numerical datasets. Defer matrix algebra, advanced indexing, and performance benchmarking until the core ideas are secure.
